# Préparer et lancer l'environnement

À exécuter une seule fois dans le terminal intégré de VS Code, depuis la racine du dépôt :

```bash
cd Gemma-Moderation
uv venv .venv --python 3.12
source .venv/bin/activate
uv pip install unsloth unsloth-zoo datasets huggingface-hub pandas numpy scikit-learn matplotlib seaborn tqdm ipykernel
python -m ipykernel install --user --name gemma-moderation --display-name "Python 3.12 (Gemma Moderation)"
hf auth login
```

À chaque nouvelle session VS Code :

```bash
cd Gemma-Moderation
source .venv/bin/activate
```

Sélectionner ensuite le kernel **Python 3.12 (Gemma Moderation)** en haut à droite du notebook.

# Fine-tuning de Gemma 4 12B pour la modération de messages

Ce notebook initialise uniquement les imports nécessaires au projet. Il ne télécharge aucune donnée, ne charge aucun modèle et ne lance aucun entraînement.

> L'entraînement sera réalisé directement dans VS Code/Jupyter avec **Unsloth** et son backend MLX natif pour Apple Silicon. Le notebook couvrira la préparation des données, le fine-tuning et l'évaluation.

**Checkpoint retenu :** [`unsloth/gemma-4-12b-it`](https://huggingface.co/unsloth/gemma-4-12b-it), au format Safetensors pour le fine-tuning. Les variantes `GGUF`, `NVFP4` et `bnb-4bit` ne seront pas utilisées pour l'entraînement local MLX.

In [ ]:
# Bibliothèque standard
import json
import os
import random
from pathlib import Path

# Manipulation et visualisation des données
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from tqdm.auto import tqdm

# Hugging Face : dataset et authentification
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import login

# Évaluation du modérateur
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

# Backend Apple Silicon
import mlx.core as mx

# API publique Unsloth ; le backend MLX est détecté automatiquement sur Mac
from unsloth import (
    FastModel,
    UnslothTrainer,
    UnslothTrainingArguments,
    is_bfloat16_supported,
)